# SQLite Primer: Storing data for a Harbour Terminal

In our harbour terminal we have a lot of data to keep track of: ships arrive and berth at the terminal, containers arrive by truck/train and are temporarily stored on the wharf before being loaded onto the ship. Likewise, containers can also be unloaded from the ship. The appropriate choice is to store that data in a database.

For relational databases, a commonly used query language is SQL (Structured Query Language). You'll find many SQL client/server databases: MySQL, Oracle, PostgreSQL,...

Typically, such databases strive for scalability and concurrent access, are hosted on a server and require access controls for the many clients connecting to them. For our uses, this might complicate things more than necessary. Nonetheless we do want you to gain some experience with SQL. That's why for this lab, we'll make use of SQLite, a small C-language library that implements an SQL database engine. No server necessary, and the data is stored right on your local computer.

## Goal

The goal of this lab is that you get familiar with SQLite and the Python `sqlite3` API. In doing so, you'll create Python functions that you can reuse later on in your project to:
- Create SQLite tables with column names and constraints
- Query, insert, and update data, whilst adhering to simple rules
- Create visualizations of the harbour terminal (in a later exercise)

## Setting up a connection

The first step is to set up a connection to the database file. A function to do that is given below.

In [7]:
# imports and variables
import sqlite3
from pathlib import Path

DB_PATH = Path('harbour-system.sqlite')

In [8]:
def get_connection(db_path=DB_PATH):
    connection = sqlite3.connect(db_path)
    connection.row_factory = sqlite3.Row
    connection.execute('PRAGMA foreign_keys = ON;')
    return connection

## 1. Creating The Schema

SQL databases follow a schema: definition of the structure of the database. The schema hence defines which tables exist, and which columns are in each of the tables, which constraints are applicable to those columns, and which keys identify rows of the table.

In our case, let's give this database schema some though:

* Ships that enter the terminal system. Whilst its being loaded and unloaded, we must consider the ship's roll and draft.
* Containers also enter the terminal system. They have a content with a weight.
* The wharf can store containers, and while a container can technically go anywhere, usually a wharf follows a grid-like pattern such that you end up with rows and columns of containers.
* The ship can also store containers, similarly to the wharf, containers are also stored in rows and columns.

Translating this to tables and columns, written as `tablename(colname0, colname1, colname2, ...)` we could follow this schema:

* `container(container_id, weight_kg)`: a table for the containers in the system.
    * `container_id`: is a unique id for the container.
    * `weight_kg`: is the weight of the container in kg.
* `ship(ship_id, roll_deg, draft_m)`: a table for the ships in the system.
    * `ship_id`: a unique id for the ship.
    * `roll_deg`: the roll of the ship in degrees.
    * `draft_m`: the deraft of the ship in meters.
* `wharf_slot(slot_id, row, col, occupancy, container_id)`: a table for the slots on the wharf.
    * `slot_id`: a unique id for the slot.
    * `row`: the physical row on the wharf, could also be called the y-position.
    * `col`: the physical column on the wharf, could also be called the x-position.
    * `occupancy`: the state of the slot, either `empty` or `occupied`.
    * `container_id`: the id of a container when `occupancy` is `occupied`.
* `ship_slot(slot_id, ship_id, row, col, occupancy, container_id)`: a table for the slots on a ship.
    * `slot_id`: a unique id for the slot.
    * `ship_id`: a unique id for the ship.
    * `row`: the physical row on the ship, could also be called the y-position.
    * `col`: the physical column on the ship, could also be called the x-position.
    * `occupancy`: the state of the slot, either `empty` or `occupied`.
    * `container_id`: the id of a container when `occupancy` is `occupied`.

### Creating Your First Table

Let's keep things simple and create the first table

In SQL you'll use the `CREATE TABLE` statement to create a table.

Typically, it follows the syntax

```
CREATE TABLE <tablename> (
    <column-name> <column-type> <column-constraints>,
    ...
);
```

Typical `column-types` are `INTEGER`, `REAL` and `TEXT`

Typical constraints are `PRIMARY KEY` or `FOREIGN KEY` constraints.
* A `PRIMARY KEY` constraint is used to uniquely identify a record/row in the table.
* A `FOREIGN KEY` constraint indicates this column entry uniquely identifies a record/row in another table.

Other common constraints are `NOT NULL` and `CHECK`
* `NOT NULL` indicates the column entry cannot be `NULL` (left empty).
* `CHECK` constraints can be used to check the column entry for various operations, e.g. equality/inequality

For example, below, we define the query that creates the `container` table.
* `container_id` is an `INTEGER` and serves as `PRIMARY KEY`.
* `weight_kg` is a `REAL`, cannot be left open with `NOT NULL`, and we check for positive weight with `CHECK (weight_kg > 0 )`

In [9]:
# Query that create a container table.
create_container_table_query = '''
CREATE TABLE IF NOT EXISTS container (
    container_id INTEGER PRIMARY KEY,
    weight_kg REAL NOT NULL CHECK (weight_kg > 0)
);
'''

Now it's up to you: create the query for the `ship` table

Keep in mind that:

* `ship_id`: is a unique id for the ship. Perfect as `PRIMARY KEY`
* `roll_deg`: the roll of the ship in degrees. Is a `REAL` value and cannot be left open (`NOT NULL`).
* `draft_m`: the draft of the ship in meters. Is also a `REAL` value and cannot be left open (`NOT NULL`). Also must be a value greater than or equal to 0, otherwise the ship'd be floating above the water (`CHECK`).

In [10]:
# Query that creates the ship table
create_ship_table_query = '''        
CREATE TABLE IF NOT EXISTS ship (
    TODO - complete the query
);
'''
### BEGIN SOLUTION
create_ship_table_query = '''        
CREATE TABLE IF NOT EXISTS ship (
    ship_id INTEGER PRIMARY KEY,
    roll_deg REAL NOT NULL,
    draft_m REAL NOT NULL CHECK (draft_m >= 0)
);
'''
### END SOLUTION

Creating the two slot tables is a bit more involved, so we'll hand them to you. Some noteworthy things:

* For the `occupancy`, we check if the the given value is in the set `('empty', 'occupied')`
* `container_id` is a `FOREIGN KEY`, as it identifies an entry in the `container` table.
* We add a check that enforces a container is presented when `occupancy` is `occupied`, or that none is present when `occupancy` is `empty`.
* We add a check that each slot has a unique combination of `row` and `col`
* For `ship_slot` we use a composite `PRIMARY KEY`. Every ship of the same make can have the same id for each of its slots, so `slot_id` is not sufficient to uniquely identify an entry in the table. Additionally adding the `ship_id` does make the key unique.

Then, after creating those tables we still have one problem:
* A `container_id` is `UNIQUE` on the wharf, and in the ship, but nothing prevents us from assigning the `container_id` to the wharf and the ship simulataneously. We can use triggers that check for this illegality, which we've given as they are also a bit involved.

In [15]:
create_ship_slot_table_query = '''        
CREATE TABLE IF NOT EXISTS wharf_slot (
    slot_id INTEGER PRIMARY KEY,
    row INTEGER NOT NULL,
    col INTEGER NOT NULL,
    occupancy TEXT NOT NULL CHECK (occupancy IN ('empty', 'occupied')),
    container_id INTEGER UNIQUE,
    FOREIGN KEY (container_id) REFERENCES container(container_id),
    CHECK (
        (occupancy = 'empty' AND container_id IS NULL)
        OR
        (occupancy = 'occupied' AND container_id IS NOT NULL)
    ),
    UNIQUE (row, col)
);
'''

create_wharf_slot_table_query = '''
CREATE TABLE IF NOT EXISTS ship_slot (
    slot_id INTEGER NOT NULL,
    ship_id INTEGER NOT NULL,
    row INTEGER NOT NULL,
    col INTEGER NOT NULL,
    occupancy TEXT NOT NULL CHECK (occupancy IN ('empty', 'occupied')),
    container_id INTEGER UNIQUE,
    PRIMARY KEY (ship_id, slot_id),
    FOREIGN KEY (ship_id) REFERENCES ship(ship_id),
    FOREIGN KEY (container_id) REFERENCES container(container_id),
    CHECK (
        (occupancy = 'empty' AND container_id IS NULL)
        OR
        (occupancy = 'occupied' AND container_id IS NOT NULL)
    ),
    UNIQUE (ship_id, row, col)
);
'''

create_wharf_slot_cross_presence_trigger_query = '''
CREATE TRIGGER IF NOT EXISTS trg_wharf_container_not_on_ship_insert
BEFORE INSERT ON wharf_slot
FOR EACH ROW
WHEN NEW.occupancy = 'occupied' AND NEW.container_id IS NOT NULL
BEGIN
    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM ship_slot
            WHERE container_id = NEW.container_id
        ) THEN RAISE(ABORT, 'Container already assigned to ship_slot')
    END;
END;

CREATE TRIGGER IF NOT EXISTS trg_wharf_container_not_on_ship_update
BEFORE UPDATE OF container_id, occupancy ON wharf_slot
FOR EACH ROW
WHEN NEW.occupancy = 'occupied' AND NEW.container_id IS NOT NULL
BEGIN
    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM ship_slot
            WHERE container_id = NEW.container_id
        ) THEN RAISE(ABORT, 'Container already assigned to ship_slot')
    END;
END;
'''

create_ship_slot_cross_presence_trigger_query = '''
CREATE TRIGGER IF NOT EXISTS trg_ship_container_not_on_wharf_insert
BEFORE INSERT ON ship_slot
FOR EACH ROW
WHEN NEW.occupancy = 'occupied' AND NEW.container_id IS NOT NULL
BEGIN
    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM wharf_slot
            WHERE container_id = NEW.container_id
        ) THEN RAISE(ABORT, 'Container already assigned to wharf_slot')
    END;
END;

CREATE TRIGGER IF NOT EXISTS trg_ship_container_not_on_wharf_update
BEFORE UPDATE OF container_id, occupancy ON ship_slot
FOR EACH ROW
WHEN NEW.occupancy = 'occupied' AND NEW.container_id IS NOT NULL
BEGIN
    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM wharf_slot
            WHERE container_id = NEW.container_id
        ) THEN RAISE(ABORT, 'Container already assigned to wharf_slot')
    END;
END;
'''

The final thing to do now is to create the schema. First stick together the 4 smaller queries, then execute the complete query with with `connection.executescript()`. The cell below this one contains some helper functions that can seed the wharf or a ship to be empty, these are given to you.

In [ ]:
def create_schema(connection):
    # TODO: create the complete query by joining the 6 queries together
    # TODO: execute the query with connection.executescript()
    ### BEGIN SOLUTION
    complete_query = create_container_table_query + \
                    create_ship_table_query + \
                    create_wharf_slot_table_query + \
                    create_ship_slot_table_query + \
                    create_wharf_slot_cross_presence_trigger_query + \
                    create_ship_slot_cross_presence_trigger_query
    connection.executescript(complete_query)
    ### END SOLUTION

In [ ]:
def seed_wharf_slots(connection, rows, cols):
    if rows <= 0 or cols <= 0:
        raise ValueError('rows and cols must be positive integers')

    slot_id = 1
    with connection:
        for row in range(1, rows + 1):
            for col in range(1, cols + 1):
                connection.execute(
                    '''
                    INSERT OR IGNORE INTO wharf_slot(slot_id, row, col, occupancy, container_id)
                    VALUES (?, ?, ?, 'empty', NULL)
                    ''',
                    (slot_id, row, col),
                )
                slot_id += 1

def seed_ship_slots(connection, ship_id, rows, cols):
    if rows <= 0 or cols <= 0:
        raise ValueError('rows and cols must be positive integers')

    slot_id = 1
    with connection:
        for row in range(1, rows + 1):
            for col in range(1, cols + 1):
                connection.execute(
                    '''
                    INSERT OR IGNORE INTO ship_slot(slot_id, ship_id, row, col, occupancy, container_id)
                    VALUES (?, ?, ?, ?, 'empty', NULL)
                    ''',
                    (slot_id, ship_id, row, col),
                )
                slot_id += 1

Let's test the function by creating this database

In [ ]:
def reset_database(db_path=DB_PATH):
    path = Path(db_path)
    if path.exists():
        path.unlink()

# reset the database to clean it.
reset_database()
with get_connection() as connection:
    create_schema(connection)
    seed_wharf_slots(connection,3,4)
    # no ship added yet, so no need to seed the ship slots.

## 2. Container Intake and Wharf Placement

With the schema ready, we can implement flows to intake containers in the system, and place them in slots on the wharf.

Keep in mind the following rules:
- A container can exist only once in table `container`.
- A slot can hold at most one container.
- `occupancy` and `container_id` must always match each other (`empty` + `NULL`, `occupied` + value).

### Inserting data in a table

Inserting into a table uses the `INSERT INTO` statement.

Typically, it looks like this: `INSERT INTO <tablename>(col0, col1, ...) VALUES (val_col0, val_col1,...)`
With the `connection.execute` statement, we can also put `?` instead of the values `(val_col0, val_col1,...)`, and pass a tuple with variables as second argument. Those `?` will then be substituted by those values.

In SQL, a write to the database is transactional, first you execute a query, then you commit the changes or if an exception occurred you rollback. In python we can automate the commit by using a `with` statement, if no exception occurs in the statement, the changes are automatically commited, and if an exception does occur they are rolled back.

The code below gives an example of how to intake a new container.

In [2]:
def intake_container(connection, container_id, weight_kg):
    with connection:
        connection.execute(
            'INSERT INTO container(container_id, weight_kg) VALUES (?, ?)',
            (container_id, weight_kg),
        )

### Reading data from a table

Reading data uses the `SELECT` statement

Typically it looks like this: `SELECT col1, col2, ... FROM <table> WHERE <selection statements> ORDER BY <ordering>`.
You specify which columns to select from the table, filter with selection statements after `WHERE` and order the results with `ORDER BY`.

Also commonly used is the `SELECT 1 FROM <table> WHERE <selection statements>` statement. It doesn't return the contents of the columns, but just a `1` for each row where the selection statements hold true. It's thus useful to check if a certain element exists.

The same remark about using `?` in the statement holds true here as well.

After executing the statement, we need to fetch the results with `.fetchall()` or fetch one row of the results with `.fetchone()`.

The code below demonstrates for example how to get all the empty slots on the wharf and their row/col pair.

In [3]:
def get_empty_wharf_slots(connection):
    rows = connection.execute(
        '''
        SELECT slot_id, row, col, occupancy, container_id
        FROM wharf_slot
        WHERE occupancy = 'empty'
        ORDER BY row, col
        '''
    ).fetchall()
    return [dict(row) for row in rows]

The final thing to learn is how to update a existing rows. This is done with the `UPDATE` function.

This typically looks like `UPDATE <table> SET col1 = val_col1, col2 = val_col2, ... WHERE <selection filter>`.
If you filter on the exact primary key in the selection filter, you obtain one row (if it exists).

The function below demonstrates how to place a container in a slot on the wharf. It checks that the rowcount is not 0, which indicates the slot is not empty

In [20]:
def place_container_in_wharf_slot(connection, container_id, slot_id):
    with connection:
        cursor = connection.execute(
            '''
            UPDATE wharf_slot
            SET occupancy = 'occupied', container_id = ?
            WHERE slot_id = ? AND occupancy = 'empty'
            ''',
            (container_id, slot_id),
        )
        if cursor.rowcount == 0:
            raise sqlite3.IntegrityError(
                f'Cannot place container {container_id} in slot {slot_id}: slot does not exist or is not empty'
            )

Let's try out these functions, and see how the constraints in the database schema are applied.

In [ ]:
# Demo: successful intake + common error cases
# reset the database
reset_database()
with get_connection() as connection:
    # create schema and seed the wharf slots as empty
    create_schema(connection)
    seed_wharf_slots(connection, 2, 2)

    # Succesfully placing a container
    print('--- Success case ---')
    intake_container(connection, 200, 11250)
    place_container_in_wharf_slot(connection, 200, 1)
    print('Placed container 200 in slot 1')

    # Trying to intake a duplicate container
    # Should fail the UNIQUE constraint (primary keys automatically have the UNIQUE constraint applied)
    print('\n--- Error case: duplicate intake_container ---')
    try:
        intake_container(connection, 200, 11300)
    except sqlite3.IntegrityError as exc:
        print(f'DB error: {exc}')

    # Trying to place a container on a non-empty slot
    # This fails with our own raised error.
    print('\n--- Error case: placing into non-empty slot ---')
    intake_container(connection, 201, 9800)
    try:
        place_container_in_wharf_slot(connection, 201, 1)
    except sqlite3.IntegrityError as exc:
        print(f'Placement error: {exc}')

    # Trying to place a container that has not yet been taken in
    # Should fail with a FOREIGN KEY constraint
    print('\n--- Error case: placing a non-intaken container ---')
    try:
        place_container_in_wharf_slot(connection, 999, 2)
    except sqlite3.IntegrityError as exc:
        print(f'Container not yet taken in: {exc}')

    print('\nCurrent empty slots:')
    print(get_empty_wharf_slots(connection))

--- Success case ---
Placed container 200 in slot 1

--- Error case: duplicate intake_container ---
DB error: UNIQUE constraint failed: container.container_id

--- Error case: placing into non-empty slot ---
Placement error: Cannot place container 201 in slot 1: slot does not exist or is not empty

--- Error case: placing a non-intaken container ---
Placement error: FOREIGN KEY constraint failed

Current empty slots:
[{'slot_id': 2, 'row': 1, 'col': 2, 'occupancy': 'empty', 'container_id': None}, {'slot_id': 3, 'row': 2, 'col': 1, 'occupancy': 'empty', 'container_id': None}, {'slot_id': 4, 'row': 2, 'col': 2, 'occupancy': 'empty', 'container_id': None}]


## 3. Moving Containers from Wharf to Ship

Now that you are familiar with the the three main operations and have some examples, it's your turn to write three functions: `add_ship`, `load_container_onto_ship` and `unload_container_to_wharf`.

In [ ]:
def add_ship(connection, ship_id, roll_deg, draft_m):
    with connection:
        # Complete this function to add a ship.
        ### BEGIN SOLUTION
        connection.execute(
            'INSERT INTO ship(ship_id, roll_deg, draft_m) VALUES (?, ?, ?)',
            (ship_id, roll_deg, draft_m),
        )
        ### END SOLUTION

def load_container_onto_ship(connection, container_id, ship_id, slot_id):
    with connection:
        # Complete this function to:
        # 1. Update the wharf slot containting the container to be empty again
        #       - Raise an error if the wharf slot is not currently occupied
        # 2. Update the ship slot to contain the container.
        #       - Raise an error if the ship slot is not empty
        ### BEGIN SOLUTION
        source_cursor = connection.execute(
            '''
            UPDATE wharf_slot
            SET occupancy = 'empty', container_id = NULL
            WHERE container_id = ? AND occupancy = 'occupied'
            ''',
            (container_id,),
        )
        if source_cursor.rowcount == 0:
            raise sqlite3.IntegrityError(
                f'Cannot load container {container_id}: it is not currently in an occupied wharf slot'
            )

        target_cursor = connection.execute(
            '''
            UPDATE ship_slot
            SET occupancy = 'occupied', container_id = ?
            WHERE ship_id = ? AND slot_id = ? AND occupancy = 'empty'
            ''',
            (container_id, ship_id, slot_id),
        )
        if target_cursor.rowcount == 0:
            raise sqlite3.IntegrityError(
                f'Cannot load into ship_id={ship_id}, slot_id={slot_id}: slot does not exist or is not empty'
            )
        ### END SOLUTION


def unload_container_to_wharf(connection, container_id, ship_id, wharf_slot_id):
    with connection:
        # Complete this function to:
        # 1. Update the ship slot containting the container to be empty again
        #       - Raise an error if the ship slot is not currently occupied
        # 2. Update the wharf slot to contain the container.
        #       - Raise an error if the wharf slot is not empty
        ### BEGIN SOLUTION
        source_cursor = connection.execute(
            '''
            UPDATE ship_slot
            SET occupancy = 'empty', container_id = NULL
            WHERE ship_id = ? AND container_id = ? AND occupancy = 'occupied'
            ''',
            (ship_id, container_id),
        )
        if source_cursor.rowcount == 0:
            raise sqlite3.IntegrityError(
                f'Cannot unload container {container_id}: it is not currently on ship {ship_id}'
            )

        target_cursor = connection.execute(
            '''
            UPDATE wharf_slot
            SET occupancy = 'occupied', container_id = ?
            WHERE slot_id = ? AND occupancy = 'empty'
            ''',
            (container_id, wharf_slot_id),
        )
        if target_cursor.rowcount == 0:
            raise sqlite3.IntegrityError(
                f'Cannot unload to wharf slot {wharf_slot_id}: slot does not exist or is not empty'
            )
        ### END SOLUTION

Let's try out these functions, and see how the constraints in the database schema are applied. Run the cell below to test for all the various successful and unsuccessful (un)loading operations.

The cell also demonstrates how the triggers prevent a container from being present in both the ship and the wharf at te same time.

In [42]:
# Demo: add_ship + load/unload success + failure + trigger protection

def setup_demo_state():
    reset_database()
    with get_connection() as connection:
        create_schema(connection)
        seed_wharf_slots(connection, 2, 2)

        add_ship(connection, 1, 0.0, 6.5)
        seed_ship_slots(connection, 1, 1, 2)


print('=== add_ship behavior ===')
setup_demo_state()
with get_connection() as connection:
    print('- Successful add_ship for ship_id=2')
    add_ship(connection, 2, 0.4, 5.8)
    print('OK')

    print('\n- Unsuccessful add_ship: duplicate ship_id')
    try:
        add_ship(connection, 1, 0.2, 6.1)
    except sqlite3.IntegrityError as exc:
        print(f'Expected error: {exc}')

    print('\n- Unsuccessful add_ship: invalid negative draft')
    try:
        add_ship(connection, 3, 0.1, -0.5)
    except sqlite3.IntegrityError as exc:
        print(f'Expected error: {exc}')

print('\n=== Load/unload function behavior ===')
setup_demo_state()
with get_connection() as connection:
    intake_container(connection, 301, 10000)
    place_container_in_wharf_slot(connection, 301, 1)

    print('\n- Successful load: wharf slot 1 -> ship slot 1')
    load_container_onto_ship(connection, 301, 1, 1)
    print('OK')

    print('\n- Unsuccessful load: same container again (no longer on wharf)')
    try:
        load_container_onto_ship(connection, 301, 1, 2)
    except sqlite3.IntegrityError as exc:
        print(f'Expected error: {exc}')

    intake_container(connection, 302, 9500)
    place_container_in_wharf_slot(connection, 302, 2)
    print('\n- Unsuccessful load: target ship slot already occupied')
    try:
        load_container_onto_ship(connection, 302, 1, 1)
    except sqlite3.IntegrityError as exc:
        print(f'Expected error: {exc}')

    print('\n- Successful unload: ship slot 1 -> wharf slot 3')
    unload_container_to_wharf(connection, 301, 1, 3)
    print('OK')

    print('\n- Unsuccessful unload: container not on ship anymore')
    try:
        unload_container_to_wharf(connection, 301, 1, 4)
    except sqlite3.IntegrityError as exc:
        print(f'Expected error: {exc}')

    print('\n- Unsuccessful unload: target wharf slot not empty')
    intake_container(connection, 303, 9700)
    place_container_in_wharf_slot(connection, 303, 4)
    load_container_onto_ship(connection, 303, 1, 2)
    try:
        unload_container_to_wharf(connection, 303, 1, 3)
    except sqlite3.IntegrityError as exc:
        print(f'Expected error: {exc}')

print('\n=== Trigger behavior (cross-table protection) ===')

print('\n- Trigger demo 1: prevent ship assignment when already on wharf')
setup_demo_state()
with get_connection() as connection:
    intake_container(connection, 401, 10300)
    place_container_in_wharf_slot(connection, 401, 1)
    try:
        connection.execute(
            '''
            UPDATE ship_slot
            SET occupancy = 'occupied', container_id = ?
            WHERE ship_id = ? AND slot_id = ?
            ''',
            (401, 1, 1),
        )
    except sqlite3.IntegrityError as exc:
        print(f'Expected trigger error: {exc}')

print('\n- Trigger demo 2: prevent wharf assignment when already on ship')
setup_demo_state()
with get_connection() as connection:
    intake_container(connection, 402, 10400)
    connection.execute(
        '''
        UPDATE ship_slot
        SET occupancy = 'occupied', container_id = ?
        WHERE ship_id = ? AND slot_id = ?
        ''',
        (402, 1, 1),
    )
    try:
        connection.execute(
            '''
            UPDATE wharf_slot
            SET occupancy = 'occupied', container_id = ?
            WHERE slot_id = ?
            ''',
            (402, 2),
        )
    except sqlite3.IntegrityError as exc:
        print(f'Expected trigger error: {exc}')

=== add_ship behavior ===
- Successful add_ship for ship_id=2
OK

- Unsuccessful add_ship: duplicate ship_id
Expected error: UNIQUE constraint failed: ship.ship_id

- Unsuccessful add_ship: invalid negative draft
Expected error: CHECK constraint failed: draft_m >= 0

=== Load/unload function behavior ===

- Successful load: wharf slot 1 -> ship slot 1
OK

- Unsuccessful load: same container again (no longer on wharf)
Expected error: Cannot load container 301: it is not currently in an occupied wharf slot

- Unsuccessful load: target ship slot already occupied
Expected error: Cannot load into ship_id=1, slot_id=1: slot does not exist or is not empty

- Successful unload: ship slot 1 -> wharf slot 3
OK

- Unsuccessful unload: container not on ship anymore
Expected error: Cannot unload container 301: it is not currently on ship 1

- Unsuccessful unload: target wharf slot not empty
Expected error: Cannot unload to wharf slot 3: slot does not exist or is not empty

=== Trigger behavior (cro

## 4. Updating Ship State and Query Helpers

Now that you're an expert, let's implement some more help functions to:
* Update `ship.roll_deg` and `ship.draft_m`
* Return terminal and ship slot states
* Locate where a specific container currently is

In [ ]:
def update_ship_state(connection, ship_id, roll_deg, draft_m):
    # Function that sets roll and draft of ship
    ### BEGIN SOLUTION
    with connection:
        cursor = connection.execute(
            '''
            UPDATE ship
            SET roll_deg = ?, draft_m = ?
            WHERE ship_id = ?
            ''',
            (roll_deg, draft_m, ship_id),
        )
        if cursor.rowcount == 0:
            raise ValueError(f'Unknown ship_id: {ship_id}')
    ### END SOLUTION

def list_wharf_state(connection):
    # Function that gets the entire wharf state as a list of dictionaries
    # Order by row then column
    ### BEGIN SOLUTION
    rows = connection.execute(
        '''
        SELECT slot_id, row, col, occupancy, container_id
        FROM wharf_slot
        ORDER BY row, col
        '''
    ).fetchall()
    return [dict(row) for row in rows]
    ### END SOLUTION


def list_ship_state(connection, ship_id):
    # Function that gets the entire ship state as a list of dictionaries
    # Order by row then column
    ### BEGIN SOLUTION
    rows = connection.execute(
        '''
        SELECT ship_id, slot_id, row, col, occupancy, container_id
        FROM ship_slot
        WHERE ship_id = ?
        ORDER BY row, col
        '''
        , (ship_id,),
    ).fetchall()
    return [dict(row) for row in rows]
    ### END SOLUTION


def find_container(connection, container_id):
    """ Function that finds a container in both the wharf and the ship.
    First, check if the container is in the terminal, return the location if it is
    Second, check if the container is in the ship, return the location if it is
    Return a dictionary with the following form:

    return {
            'location': 'ship_slot',
            'ship_id': None or match['ship_id'],
            'slot_id': match['slot_id'],
            'row': match['row'],
            'col': match['col'],
        } 

    HINT: match = connection.execute(your statement).fetchone()    
    """
    ### BEGIN SOLUTION
    terminal_match = connection.execute(
        '''
        SELECT slot_id, row, col
        FROM wharf_slot
        WHERE container_id = ?
        '''
        , (container_id,),
    ).fetchone()
    if terminal_match is not None:
        return {
            'location': 'wharf_slot',
            'ship_id': None,
            'slot_id': terminal_match['slot_id'],
            'row': terminal_match['row'],
            'col': terminal_match['col'],
        }

    ship_match = connection.execute(
        '''
        SELECT ship_id, slot_id, row, col
        FROM ship_slot
        WHERE container_id = ?
        '''
        , (container_id,),
    ).fetchone()
    if ship_match is not None:
        return {
            'location': 'ship_slot',
            'ship_id': ship_match['ship_id'],
            'slot_id': ship_match['slot_id'],
            'row': ship_match['row'],
            'col': ship_match['col'],
        }

    return None
    ### END SOLUTION

Let's try out these functions, the cell below defines a bunch of demo tests that try out your functions


In [40]:
# Demo tests for: update_ship_state, list_wharf_state, list_ship_state, find_container
reset_database()
with get_connection() as connection:
    create_schema(connection)
    seed_wharf_slots(connection, 3, 4)

    add_ship(connection, 1, 0.0, 6.5)
    seed_ship_slots(connection, 1, 1, 3)

    intake_container(connection, 501, 10100)
    intake_container(connection, 502, 9800)
    place_container_in_wharf_slot(connection, 501, 1)
    place_container_in_wharf_slot(connection, 502, 2)

    load_container_onto_ship(connection, 501, 1, 1)

    update_ship_state(connection, 1, roll_deg=2.2, draft_m=7.3)
    ship_row = connection.execute(
        'SELECT roll_deg, draft_m FROM ship WHERE ship_id = ?',
        (1,),
    ).fetchone()
    assert ship_row['roll_deg'] == 2.2
    assert ship_row['draft_m'] == 7.3
    print('✅ update_ship_state test passed')

    wharf_state = list_wharf_state(connection)
    assert len(wharf_state) == 12
    wharf_slot_1 = [slot for slot in wharf_state if slot['slot_id'] == 1][0]
    assert wharf_slot_1['occupancy'] == 'empty'
    assert wharf_slot_1['container_id'] is None
    print('✅ list_wharf_state test passed')

    ship_state = list_ship_state(connection, 1)
    assert len(ship_state) == 3
    ship_slot_1 = [slot for slot in ship_state if slot['slot_id'] == 1][0]
    assert ship_slot_1['occupancy'] == 'occupied'
    assert ship_slot_1['container_id'] == 501
    print('✅ list_ship_state test passed')

    found_501 = find_container(connection, 501)
    assert found_501 == {
        'location': 'ship_slot',
        'ship_id': 1,
        'slot_id': 1,
        'row': 1,
        'col': 1,
    }

    found_502 = find_container(connection, 502)
    assert found_502 == {
        'location': 'wharf_slot',
        'ship_id': None,
        'slot_id': 2,
        'row': 1,
        'col': 2,
    }

    found_missing = find_container(connection, 9999)
    assert found_missing is None
    print('✅ find_container tests passed')

print('All helper-function tests passed ✅')

✅ update_ship_state test passed
✅ list_wharf_state test passed
✅ list_ship_state test passed
✅ find_container tests passed
All helper-function tests passed ✅


## 5. Practice scenario

Use your functions to complete this scenario:

1. Create schema and seed a `3 x 4` terminal grid.
2. Add one ship with ID `1` and initial `roll_deg=0`, `draft_m=6.5`.
3. Intake three containers (`100`, `101`, `102`) with realistic weights. 
4. Place them in wharf slots `1`,`2`,`3` respectively.
5. Prepare the `ship_slot` as a `1 x 2` grid for ship `1`.
6. Load container `101` from terminal to ship and update ship state.
7. Query terminal and ship state and verify where each container is.

In [39]:
def setup_practice_scenario():
    # Write your function that sets up the practice scenario
    ### BEGIN SOLUTION
    with get_connection() as connection:
        # 1) Create schema and seed a 3x4 terminal grid
        create_schema(connection)
        seed_wharf_slots(connection, 3, 4)

        # 2) Add one ship with ID 1 and initial state
        add_ship(connection, 1, 0.0, 6.5)

        # 5) Prepare and fill at least two ship slots for ship 1
        seed_ship_slots(connection, 1, 1, 2)

        # 3) Intake three containers
        intake_container(connection, 100, 10200)
        intake_container(connection, 101, 9800)
        intake_container(connection, 102, 11050)

        # 4) Place them in terminal slots
        place_container_in_wharf_slot(connection, 100, 1)
        place_container_in_wharf_slot(connection, 101, 2)
        place_container_in_wharf_slot(connection, 102, 3)

        # 6) Load one container to ship and update ship state
        load_container_onto_ship(connection, 101, 1, 1)
        update_ship_state(connection, 1, roll_deg=1.3, draft_m=6.9)

        # 7) Query terminal/ship state and verify container locations
        terminal_state = list_wharf_state(connection)
        ship_state = list_ship_state(connection, 1)

        print('Terminal state:')
        for row in terminal_state:
            print(row)

        print('\nShip state (ship_id=1):')
        for row in ship_state:
            print(row)

        print('\nContainer locations:')
        for container_id in (100, 101, 102):
            print(container_id, '->', find_container(connection, container_id))

        assert find_container(connection, 100)['location'] == 'wharf_slot'
        assert find_container(connection, 101)['location'] == 'ship_slot'
        assert find_container(connection, 102)['location'] == 'wharf_slot'
        print('\n✅ Practice scenario completed successfully')
    ### END SOLUTION

# Reset database and call your function to setup up the database.
reset_database()
setup_practice_scenario()


Terminal state:
{'slot_id': 1, 'row': 1, 'col': 1, 'occupancy': 'occupied', 'container_id': 100}
{'slot_id': 2, 'row': 1, 'col': 2, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 3, 'row': 1, 'col': 3, 'occupancy': 'occupied', 'container_id': 102}
{'slot_id': 4, 'row': 1, 'col': 4, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 5, 'row': 2, 'col': 1, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 6, 'row': 2, 'col': 2, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 7, 'row': 2, 'col': 3, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 8, 'row': 2, 'col': 4, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 9, 'row': 3, 'col': 1, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 10, 'row': 3, 'col': 2, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 11, 'row': 3, 'col': 3, 'occupancy': 'empty', 'container_id': None}
{'slot_id': 12, 'row': 3, 'col': 4, 'occupancy': 'empty', 'container_id': None}

Ship state (ship_id=1):
{'sh

In [ ]:
# Grading cell for the practice scenario. Execute this cell to check that you implemented the practice scenario correctly.
reset_database()
setup_practice_scenario()

with get_connection() as connection:
    ship = connection.execute(
        'SELECT roll_deg, draft_m FROM ship WHERE ship_id = 1'
    ).fetchone()
    assert ship is not None, 'Ship with ship_id=1 should exist after setup_practice_scenario'
    assert ship['roll_deg'] == 1.3, 'Ship roll_deg should be 1.3 after practice scenario'
    assert ship['draft_m'] == 6.9, 'Ship draft_m should be 6.9 after practice scenario'

    wharf_slot_1 = connection.execute(
        'SELECT occupancy, container_id FROM wharf_slot WHERE slot_id = 1'
    ).fetchone()
    wharf_slot_2 = connection.execute(
        'SELECT occupancy, container_id FROM wharf_slot WHERE slot_id = 2'
    ).fetchone()
    wharf_slot_3 = connection.execute(
        'SELECT occupancy, container_id FROM wharf_slot WHERE slot_id = 3'
    ).fetchone()

    assert wharf_slot_1['occupancy'] == 'occupied', 'Wharf slot 1 should be occupied'
    assert wharf_slot_1['container_id'] == 100, 'Wharf slot 1 should contain container 100'

    assert wharf_slot_2['occupancy'] == 'empty', 'Wharf slot 2 should be empty after loading container 101'
    assert wharf_slot_2['container_id'] is None, 'Wharf slot 2 container_id should be NULL'

    assert wharf_slot_3['occupancy'] == 'occupied', 'Wharf slot 3 should be occupied'
    assert wharf_slot_3['container_id'] == 102, 'Wharf slot 3 should contain container 102'

    ship_slot_1 = connection.execute(
        'SELECT occupancy, container_id FROM ship_slot WHERE ship_id = 1 AND slot_id = 1'
    ).fetchone()
    ship_slot_2 = connection.execute(
        'SELECT occupancy, container_id FROM ship_slot WHERE ship_id = 1 AND slot_id = 2'
    ).fetchone()

    assert ship_slot_1['occupancy'] == 'occupied', 'Ship slot 1 should be occupied after loading'
    assert ship_slot_1['container_id'] == 101, 'Ship slot 1 should contain container 101'
    assert ship_slot_2['occupancy'] == 'empty', 'Ship slot 2 should remain empty'
    assert ship_slot_2['container_id'] is None, 'Ship slot 2 container_id should be NULL'

    location_100 = find_container(connection, 100)
    location_101 = find_container(connection, 101)
    location_102 = find_container(connection, 102)

    assert location_100['location'] == 'wharf_slot', 'Container 100 should be on the wharf'
    assert location_101['location'] == 'ship_slot', 'Container 101 should be on the ship'
    assert location_102['location'] == 'wharf_slot', 'Container 102 should be on the wharf'

print('✅ Grade check for practice scenario passed')

## Addtional Grading Checks

These checks are used for additional grading besides the grading of the practice scenario.

In [46]:
# Grade check 1: schema + helper seeders + triggers
grading_db_path = Path('harbour-system-grading.sqlite')
if grading_db_path.exists():
    grading_db_path.unlink()

with get_connection(grading_db_path) as connection:
    create_schema(connection)
    seed_wharf_slots(connection, 2, 3)

    tables = {
        row['name']
        for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    }
    assert {'container', 'ship', 'wharf_slot', 'ship_slot'}.issubset(tables), 'Missing expected tables'

    triggers = {
        row['name']
        for row in connection.execute("SELECT name FROM sqlite_master WHERE type='trigger'").fetchall()
    }
    assert {
        'trg_wharf_container_not_on_ship_insert',
        'trg_wharf_container_not_on_ship_update',
        'trg_ship_container_not_on_wharf_insert',
        'trg_ship_container_not_on_wharf_update',
    }.issubset(triggers), 'Missing expected cross-table triggers'

    total_wharf_slots = connection.execute('SELECT COUNT(*) AS n FROM wharf_slot').fetchone()['n']
    empty_wharf_slots = connection.execute("SELECT COUNT(*) AS n FROM wharf_slot WHERE occupancy='empty'").fetchone()['n']
    assert total_wharf_slots == 6, 'seed_wharf_slots should create rows*cols wharf slots'
    assert empty_wharf_slots == 6, 'All seeded wharf slots should be empty'

    add_ship(connection, 99, 0.0, 5.0)
    seed_ship_slots(connection, 99, 1, 3)
    total_ship_slots = connection.execute(
        'SELECT COUNT(*) AS n FROM ship_slot WHERE ship_id = 99'
    ).fetchone()['n']
    empty_ship_slots = connection.execute(
        "SELECT COUNT(*) AS n FROM ship_slot WHERE ship_id = 99 AND occupancy='empty'"
    ).fetchone()['n']
    assert total_ship_slots == 3, 'seed_ship_slots should create rows*cols ship slots'
    assert empty_ship_slots == 3, 'All seeded ship slots should be empty'

print('✅ Grade check 1 passed')

✅ Grade check 1 passed


In [47]:
# Grade check 2: load + unload + update_ship_state + list helpers
grading_db_path = Path('harbour-system-grading.sqlite')
if grading_db_path.exists():
    grading_db_path.unlink()

with get_connection(grading_db_path) as connection:
    create_schema(connection)
    seed_wharf_slots(connection, 2, 3)
    add_ship(connection, 1, 0.0, 6.5)
    seed_ship_slots(connection, 1, 1, 3)

    intake_container(connection, 101, 9800)
    place_container_in_wharf_slot(connection, 101, 2)

    load_container_onto_ship(connection, 101, 1, 1)
    on_ship = connection.execute(
        "SELECT occupancy, container_id FROM ship_slot WHERE ship_id=1 AND slot_id=1"
    ).fetchone()
    src_terminal = connection.execute(
        "SELECT occupancy, container_id FROM wharf_slot WHERE slot_id=2"
    ).fetchone()
    assert on_ship['occupancy'] == 'occupied', 'Ship slot should be occupied after load'
    assert on_ship['container_id'] == 101, 'Ship slot should reference loaded container'
    assert src_terminal['occupancy'] == 'empty', 'Source terminal slot should be empty after load'
    assert src_terminal['container_id'] is None, 'Source terminal slot should clear container_id after load'

    update_ship_state(connection, 1, roll_deg=1.8, draft_m=7.1)
    ship = connection.execute("SELECT roll_deg, draft_m FROM ship WHERE ship_id=1").fetchone()
    assert ship['roll_deg'] == 1.8, 'Ship roll_deg should be updated'
    assert ship['draft_m'] == 7.1, 'Ship draft_m should be updated'

    unload_container_to_wharf(connection, 101, 1, 4)
    back_terminal = connection.execute(
        "SELECT occupancy, container_id FROM wharf_slot WHERE slot_id=4"
    ).fetchone()
    emptied_ship = connection.execute(
        "SELECT occupancy, container_id FROM ship_slot WHERE ship_id=1 AND slot_id=1"
    ).fetchone()
    assert back_terminal['occupancy'] == 'occupied', 'Target terminal slot should be occupied after unload'
    assert back_terminal['container_id'] == 101, 'Terminal slot should reference unloaded container'
    assert emptied_ship['occupancy'] == 'empty', 'Ship slot should be empty after unload'
    assert emptied_ship['container_id'] is None, 'Ship slot should clear container_id after unload'

    terminal_state = list_wharf_state(connection)
    ship_state = list_ship_state(connection, 1)
    assert len(terminal_state) == 6, 'list_wharf_state should return all terminal slots'
    assert len(ship_state) == 3, 'list_ship_state should return all slots for the ship'

print('✅ Grade check 2 passed')

✅ Grade check 2 passed
